# Modelo Machine Learning

Este notebook va a consistir en diseñar varios modelos con tareas independientes cada uno. Estas tareas serán:

- Clasificación: Predecir si un estudiante está en riesgo académico. (Target binario)

- Regresión: Predecir la nota exacta de un estudiante. (Target continuo)

- Clustering: Segmentar estudiante en perfiles interpretables. (No se utiliza target)

## 01 - Carga y preprocesamiento

Importamos librerías, cargamos el dataset y realizamos algunos ajustes a los datos.

In [1]:
# Importación de librerías.
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings 

# Librerías de ML.
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve,
                             mean_squared_error, mean_absolute_error, r2_score)
from xgboost import XGBClassifier, XGBRegressor

# Configuración de librerías.
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print(f'Librerías cargados correctamente.')

Librerías cargados correctamente.


In [2]:
# Carga del dataset.
ruta = 'data\GAP_clean.csv'
df = pd.read_csv(ruta)

# Restauramos los tipos ordinales.
stress_order = pd.CategoricalDtype(categories=['Low', 'Medium', 'High'], ordered=True)
df['stress_level'] = df['stress_level'].astype(stress_order)

intensity_order = pd.CategoricalDtype(
    categories=['Mínimo (0–1h)', 'Moderado (1–3h)', 'Intenso (3–6h)', 'Extremo (6–8h)'],
    ordered=True
)
df['gaming_intensity'] = df['gaming_intensity'].astype(intensity_order)

print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
df.head()

Dataset cargado: 8,000 filas × 16 columnas


,student_id,age,gender,gaming_hours,study_hours,sleep_hours,attendance,gaming_genre,social_activity,device_usage,reaction_time_ms,addiction_score,stress_level,grades,risk_flag,gaming_intensity
0,1,22,Male,7.23,8.78,6.96,91.44,FPS,3.25,9.36,235.84,14.69,Low,86.459555,0,NaN
1,2,19,Male,0.07,8.72,7.63,63.63,Casual,1.02,3.21,328.71,2.47,Medium,98.230000,0,NaN
2,3,23,Female,1.73,9.56,4.40,83.26,Casual,3.46,5.56,313.61,4.73,High,90.560000,0,NaN
3,4,20,Female,6.62,1.68,7.83,75.04,RPG,1.46,11.78,241.84,14.54,Low,32.670000,1,NaN
4,5,22,Female,5.36,5.83,5.55,65.57,FPS,1.01,8.23,249.31,12.48,Low,58.710000,0,NaN


Tras importar las librerías y cargar el dataset, comenzamos el preprocesamiento.

En esta sección, se prepararan las variables para que el modelo pueda aprender de ellas lo mejor posible.

En este caso, comenzaremos por codificar las variables categóricas como son el género, el tipo de juego o el nivel de estrés.

In [4]:
# Codificación de variables categóricas.
le = LabelEncoder() 

df['gender_enc'] = le.fit_transform(df['gender'])
df['gaming_genre_enc'] = le.fit_transform(df['gaming_genre'])
df['stress_level_enc'] = le.fit_transform(df['stress_level'])

print('Variables categóricas codificadas correctamente.')

Variables categóricas codificadas correctamente.


In [8]:
# Selección de características.
features = ['age', 'gaming_hours', 'study_hours', 'sleep_hours',
    'attendance', 'social_activity', 'device_usage',
    'reaction_time_ms', 'addiction_score',
    'gender_enc', 'gaming_genre_enc', 'stress_level_enc']

# Variable objetivo de los modelos.
target_class = 'risk_flag'
target_reg = 'grades'

# Características.
X = df[features]

print('Features seleccionadas:')
print(features)
print(f'\nEstructura de X: {X.shape}')

Features seleccionadas:
['age', 'gaming_hours', 'study_hours', 'sleep_hours', 'attendance', 'social_activity', 'device_usage', 'reaction_time_ms', 'addiction_score', 'gender_enc', 'gaming_genre_enc', 'stress_level_enc']

Estructura de X: (8000, 12)


Se ha utilizado LabelEncoder debido a que en se van a utilizar los modelos XGBoost y RandomForest, que manejan bien las varibles numéricas ordinales.

Para una regresión logística sería más apropiado OneHotEncoder, pero como en este caso hay pocas categorías con poco impacto, se ha utilizado LabelEncoder.

A continuación, se dividirá entre conjuntos de Train/Test.

In [ ]:
# Modelo de clasificación.
y_class = df[target_class]
X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(X, y_class, test_size=0.2, random_state=42, 
                                                                            stratify=y_class) # Stratify para mantener la proporción de clases entre train y test.


# Modelo de regresión.
y_reg = df[target_reg]
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)

# Mostramos la distribución.
print(f'Clasificación  — Train: {X_train_class.shape[0]:,} | Test: {X_test_class.shape[0]:,}')
print(f'Regresión      — Train: {X_train_reg.shape[0]:,} | Test: {X_test_reg.shape[0]:,}')
print(f'\nBalance en train — riesgo: {y_train_class.mean():.1%}')
print(f'Balance en test  — riesgo: {y_test_class.mean():.1%}')

Clasificación  — Train: 6,400 | Test: 1,600
Regresión      — Train: 6,400 | Test: 1,600

Balance en train — riesgo: 25.1%
Balance en test  — riesgo: 25.1%


Tras dividir los datasets para cada modelo, continuaremos escalando las variables para los modelos de regresión (logística y lineal).

Estos modelos son sensibles a la escala debido a la optimización mediante gradiente que aplican. 

En cambio, RandomForest y XGBoost aprenden mediante umbrales, por tanto no es necesario escalar.

In [ ]:
# Scaler.
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_reg) # Solo se ajusta con el train para evitar data leakage.
X_test_scaled = scaler.transform(X_test_reg) # Se transforma con el mismo scaler ajustado al train.

print('Escalado aplicado. Medias en train (deben ser ~0):')
print(pd.Series(X_train_scaled.mean(axis=0), index=features).round(4))

Escalado aplicado. Medias en train (deben ser ~0):
age                -0.0
gaming_hours        0.0
study_hours         0.0
sleep_hours         0.0
attendance         -0.0
social_activity    -0.0
device_usage       -0.0
reaction_time_ms   -0.0
addiction_score    -0.0
gender_enc         -0.0
gaming_genre_enc    0.0
stress_level_enc   -0.0
dtype: float64


## 02 - Clasificación